<a href="https://colab.research.google.com/github/nosadchiy/public/blob/main/NonStationary_Inventory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Compute order up to levels for time-dependent stochastic demand. Zero lead time, zero fixed ordering costs.

In [10]:
# Nikolay Osadchiy, 2019, 2021 (ported from MATLAB to Python)

import numpy as np
from scipy.stats import poisson


def cost_per_q_vectorized(b: float, h: float, lam: float, opt_cost_to_go: np.ndarray) -> np.ndarray:
    """
    Vectorized expected cost for all q=0..N in one shot.
    Returns array cost_q of shape (N+1,).
    """
    T = int(len(opt_cost_to_go))          # T = N+1
    demand = np.arange(T)                # 0..N

    P = poisson.pmf(demand, lam)
    P = P / P.sum()                      # match MATLAB normalization

    q = np.arange(T)[:, None]            # (T,1)
    d = demand[None, :]                  # (1,T)

    inv_end = np.maximum(q - d, 0).astype(int)           # (T,T) in [0..N]

    holding = h * inv_end
    backorder = b * np.maximum(d - q, 0)

    continuation = opt_cost_to_go[inv_end]               # fancy indexing (T,T)

    per_scenario_cost = holding + backorder + continuation  # (T,T)

    # expectation over demand
    return (per_scenario_cost @ P).astype(float)         # (T,)


def suffix_argmin(cost_q: np.ndarray):
    """
    For each x, compute:
      best_q[x] = argmin_{q >= x} cost_q[q]
      best_cost[x] = min_{q >= x} cost_q[q]
    O(N) backward pass.
    """
    N = len(cost_q) - 1
    best_q = np.empty(N + 1, dtype=int)
    best_cost = np.empty(N + 1, dtype=float)

    best_q[N] = N
    best_cost[N] = cost_q[N]

    for x in range(N - 1, -1, -1):
        if cost_q[x] <= best_cost[x + 1]:
            best_q[x] = x
            best_cost[x] = cost_q[x]
        else:
            best_q[x] = best_q[x + 1]
            best_cost[x] = best_cost[x + 1]

    return best_q, best_cost


def solve_order_up_to_policy_fast(lambda_vec, b=9.0, h=1.0):
    lambda_vec = np.asarray(lambda_vec, dtype=float)
    n = len(lambda_vec)

    N = int(4 * np.max(lambda_vec))
    beta = b / (b + h)

    cost_opt = np.zeros((N + 1, n + 1), dtype=float)
    S_opt = np.zeros((N + 1, n + 1), dtype=int)

    for t in range(n - 1, -1, -1):
        lam = float(lambda_vec[t])
        opt_cost_to_go = cost_opt[:, t + 1]     # length N+1

        # 1) vectorized expected cost for all q
        cost_q = cost_per_q_vectorized(b, h, lam, opt_cost_to_go)  # (N+1,)

        # 2) best feasible q for each x via suffix argmin
        best_q, best_cost = suffix_argmin(cost_q)

        # 3) fill value function
        cost_opt[:, t] = best_cost

        # 4) map best_q -> best "S" (to mimic MATLAB tie-breaking):
        #    if best_q[x]==x, any S<=x gives same q=x; MATLAB argmin picks S=0.
        S_t = best_q.copy()
        S_t[best_q == np.arange(N + 1)] = 0
        S_opt[:, t] = S_t

    print("[Order-up-to Policy] the optimal inventory level for each period:")
    print(S_opt[0, :n])

    print("\n[Myopic Policy] the optimal inventory level for each period:")
    myopic = poisson.ppf(beta, lambda_vec).astype(int)
    print(myopic)

    return S_opt, cost_opt, myopic


if __name__ == "__main__":
    lambda_vec = [26, 8, 4, 2, 1]
    S_opt, cost_opt, myopic = solve_order_up_to_policy_fast(lambda_vec, b=9.0, h=1.0)

[Order-up-to Policy] the optimal inventory level for each period:
[32 11  6  4  2]

[Myopic Policy] the optimal inventory level for each period:
[33 12  7  4  2]
